In [2]:
#-----Federated Graph Learning----Share data model
"""
Imagine three hospitals participating in oncology research:
- Hospital A holds its own lung-cancer records.
- Hospital B holds its own biomarker and treatment records.
- Hospital C holds its own clinical outcomes.
The hospitals cannot simply pool their patient-level data into one central database. They want to explore whether a shared model 
can learn from all three datasets while the raw records remain local.

The general workflow is:
1. A coordinator sends the current model to each hospital.
2. Each hospital trains the model on its own local data.
3. Hospitals send model updates—not their raw patient records—to the coordinator.
4. The coordinator aggregates the updates and sends back the improved shared model.
5. The process repeats over several rounds.

Python program -- We'll use a very simple linear model. Each hospital has a graph-derived feature representing the number 
of documented biomarker-to-treatment links for a case, and a numeric outcome score.
The model learns the relationship between that feature and the outcome.
How to understand it
- Each hospital calculates gradients from its own data.
- The coordinator aggregates those gradients, weighted by the number of records at each hospital.
- The shared model parameters are updated.
- After 100 rounds, the model should learn an approximate relationship between the graph-derived feature and the outcome.
"""
# Illustrative federated learning simulation.# Each hospital retains its own local training records.
# Only model gradients are sent to the coordinator.

hospitals = {
    "Hospital_A": [
        (1.0, 2.0),
        (2.0, 4.0),
        (3.0, 6.0)
    ],
    "Hospital_B": [
        (1.0, 3.0),
        (2.0, 5.0),
        (3.0, 7.0)
    ],
    "Hospital_C": [
        (1.0, 4.0),
        (2.0, 6.0),
        (3.0, 8.0)
    ]
}
# ------ Shared model: predicted outcome = w * feature + b
w = 0.0
b = 0.0
learning_rate = 0.01
rounds = 100

def calculate_local_gradients(records, w, b):
    grad_w = 0.0
    grad_b = 0.0
    for x, actual in records:
        predicted = w * x + b
        error = predicted - actual
        grad_w += 2 * error * x
        grad_b += 2 * error
    count = len(records)
    return grad_w / count, grad_b / count

for round_number in range(rounds):
    total_grad_w = 0.0
    total_grad_b = 0.0
    total_records = 0
    
    for records in hospitals.values():                          # Each hospital calculates gradients locally.
        local_w, local_b = calculate_local_gradients(records, w, b)
        count = len(records)
        total_grad_w += local_w * count                         # Coordinator aggregates weighted gradients.
        total_grad_b += local_b * count
        total_records += count

    average_grad_w = total_grad_w / total_records               # Update the shared model.
    average_grad_b = total_grad_b / total_records

    w -= learning_rate * average_grad_w
    b -= learning_rate * average_grad_b

print("Shared model weight:", round(w, 3))
print("Shared model bias:", round(b, 3))
print("Prediction for feature value 2:", round(w * 2 + b, 3))
print("Raw records remain at each hospital.")

Shared model weight: 2.035
Shared model bias: 0.921
Prediction for feature value 2: 4.99
Raw records remain at each hospital.
